# 2 - Match crashes to LION segments

Create the first analytical table: one row per LION segment with crash harm aggregated over the study period.

In [1]:
from pathlib import Path
import sys

# Make imports work whether Jupyter starts in the repo root or notebooks/.
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

CACHE_DIR = ROOT / "data" / "cache"
OUTPUT_DIR = ROOT / "data" / "outputs"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ANALYSIS_START = "2021-01-01"
ANALYSIS_END = "2025-12-31"
REFRESH = False

print("Repo root:", ROOT)
print("Analysis period:", ANALYSIS_START, "to", ANALYSIS_END)
print("Force API refresh:", REFRESH)


Repo root: /Users/uyen-phuongnguyen/schoolgithub/worst-streets-watchlist
Analysis period: 2021-01-01 to 2025-12-31
Force API refresh: False


In [2]:
from src.crashes import fetch_crashes, crashes_to_geodataframe
from src.lion import fetch_lion, prepare_lion_segments
from src.joins import join_crashes_to_lion, aggregate_crashes_by_segment, attach_crash_metrics
from src.api import save_cache


In [3]:
crashes = fetch_crashes(ANALYSIS_START, ANALYSIS_END, refresh=REFRESH, cache_dir=CACHE_DIR)
lion_raw = fetch_lion(refresh=REFRESH, cache_dir=CACHE_DIR)
lion_segments = prepare_lion_segments(lion_raw)
crash_points = crashes_to_geodataframe(crashes)

print(f"Geocoded crash points: {len(crash_points):,}")
print(f"LION segments: {len(lion_segments):,}")


h9gi-nx95: fetched 50,000 rows
h9gi-nx95: fetched 100,000 rows
h9gi-nx95: fetched 150,000 rows
h9gi-nx95: fetched 200,000 rows
h9gi-nx95: fetched 250,000 rows
h9gi-nx95: fetched 300,000 rows
h9gi-nx95: fetched 350,000 rows
h9gi-nx95: fetched 400,000 rows
h9gi-nx95: fetched 450,000 rows
h9gi-nx95: fetched 487,931 rows
ArcGIS: fetched 2,000 features
ArcGIS: fetched 4,000 features
ArcGIS: fetched 6,000 features
ArcGIS: fetched 8,000 features
ArcGIS: fetched 10,000 features
ArcGIS: fetched 12,000 features
ArcGIS: fetched 14,000 features
ArcGIS: fetched 16,000 features
ArcGIS: fetched 18,000 features
ArcGIS: fetched 20,000 features
ArcGIS: fetched 22,000 features
ArcGIS: fetched 24,000 features
ArcGIS: fetched 26,000 features
ArcGIS: fetched 28,000 features
ArcGIS: fetched 30,000 features
ArcGIS: fetched 32,000 features
ArcGIS: fetched 34,000 features
ArcGIS: fetched 36,000 features
ArcGIS: fetched 38,000 features
ArcGIS: fetched 40,000 features
ArcGIS: fetched 42,000 features
ArcGIS: fetch

## Nearest-segment match

The 75-foot threshold first pass. Try out later with 25, 50, 100ft increaments?. look for unmatched records.

a future intersection model should instead use LION nodes, more accurate?

In [4]:
crash_lion = join_crashes_to_lion(
    crash_points,
    lion_segments,
    max_distance_ft=75,
)

matched = crash_lion["segmentid"].notna()
print(f"Matched crashes: {matched.sum():,} / {len(crash_lion):,} ({matched.mean():.1%})")
print(crash_lion["lion_distance_ft"].describe())


Matched crashes: 445,557 / 449,611 (99.1%)
count    445557.000000
mean          5.018867
std          11.180275
min           0.000064
25%           0.805743
50%           1.760033
75%           3.306803
max          74.994777
Name: lion_distance_ft, dtype: float64


In [ ]:
# Inspect the farthest successful matches.
cols = [c for c in [
    "collision_id", "crash_date", "on_street_name", "cross_street_name",
    "segmentid", "lion_distance_ft"
] if c in crash_lion.columns]

display(crash_lion.loc[matched, cols].nlargest(20, "lion_distance_ft"))


In [ ]:
segment_crashes = aggregate_crashes_by_segment(crash_lion)
segments_with_crashes = attach_crash_metrics(lion_segments, segment_crashes)

print(f"Segments with >=1 matched crash: {(segments_with_crashes['crashes'] > 0).sum():,}")
display(
    segments_with_crashes.sort_values("people_injured", ascending=False)[
        [c for c in ["segmentid", "street", "crashes", "people_injured", "people_killed"] if c in segments_with_crashes.columns]
    ].head(20)
)


In [ ]:
save_cache(
    crash_lion,
    "crash_lion_matches",
    cache_dir=CACHE_DIR,
    metadata={"analysis_start": ANALYSIS_START, "analysis_end": ANALYSIS_END, "max_distance_ft": 75},
)
save_cache(
    segments_with_crashes,
    "segments_with_crashes",
    cache_dir=CACHE_DIR,
    metadata={"analysis_start": ANALYSIS_START, "analysis_end": ANALYSIS_END},
)
print("Save snapshots.")
